# Week 5: RAG Pipeline with Retrieval Evaluation

## Purpose
Retrieval-augmented generation is only as good as its retrieval. Retrieval quality is something you can measure rather than eyeball. This notebook builds a complete RAG pipeline end to end. Part 1 handles ingestion, chunking, embedding, and retrieval over a small corpus of Microsoft Learn documentation stored as markdown files in `Docs/`. Everything is backed by a local Postgres database running the `pgvector` extension in Docker. I built three chunking strategies. The first is a fixed-window splitter. The second is a LangChain `MarkdownHeaderTextSplitter` that breaks each document into sections along its `#`/`##`/`###` headers. The third is a header-plus-recursive-character hybrid that bounds section length. For the retrieval comparison, I held chunking constant on the markdown-header splitter. This let me isolate the effect of the embedding model. I compared a local, free `sentence-transformers` model against `text-embedding-3-small`, hosted on Microsoft Foundry.

Part 2 evaluates that retrieval. With chunking fixed on `MarkdownHeaderTextSplitter`, I defined an expanded relevant-chunk set for each of ten test queries. I compared the two embedding models using chunk-level precision and recall at k=1 and k=5. Both models were queried the same way through a single SQL-plus-Python `retrieve()` function. Part 3 connects that retrieval to an LLM-generated answer. The top retrieved chunks are formatted into a grounded prompt for an Azure Foundry-hosted model. The model was instructed to answer only from the provided context and cite the chunks it used. Part 4 examines one query for which retrieval surfaced the wrong chunk, discusses likely reasons for the ranking error, and proposes mitigations.

In [1]:
# Imports
from dataclasses import dataclass, field
import re, yaml
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from pathlib import Path
import os, re, pathlib, numpy as np
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import hashlib, json, psycopg, pgvector
from pgvector.psycopg import register_vector
from psycopg import sql
import pandas as pd


c:\Users\brett\projects\github\bretttay24\cosc-650-applied-llm-systems\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part 1: Build the pipeline
Ingest and chunk eight documents, embed the chunks with both a local sentence-transformer and a Microsoft Foundry-hosted model, and store the vectors in PostgreSQL with pgvector.

### 1a. Chunk Class

In [2]:
@dataclass
class Chunk:
    text: str
    source: str          # week-5/Docs/Neo4j.md
    chunker: str         # "md-header", "fixed-320-40", ...
    ordinal: int         # position within the doc for this chunker
    header_path: str     # "Microsoft Agent Framework Overview > Install > Python"
    meta: dict = field(default_factory=dict)   # frontmatter + anything else

### 1b. Load Documents
This uses regular expression (re) to retrieve the metadata and raw text of document located in the `week-5\Docs\` directory.
* The Front Matter regular expression is used to match YAML front matter at the beginning of a document.

Reads every `.md` under `Docs/`, asserting at least eight documents and a frontmatter `title` on each. Before embedding, the title is prepended to the text that will be embedded/vectorized.

#### Parse Metadata at Top of Documents

In [3]:
# === Define regular expressions for front matter ===
front_matter_re = re.compile(r'\A---\s*\n(.*?)\n---\s*\n', re.DOTALL)


def load_doc(path):
    """ 
    Load a document from the given path, parse its front matter and body.

    Returns a tuple (meta, body) where meta is a dictionary of front matter and body is the remaining text.
    """
    raw_text = path.read_text(encoding='utf-8')
    matched_front_matter = front_matter_re.match(raw_text)
    meta_data = yaml.safe_load(matched_front_matter.group(1)) if matched_front_matter else {}
    body = raw_text[matched_front_matter.end():] if matched_front_matter else raw_text
    return meta_data, body

# === Define a function to prepend Title and Header Path to chunk text ===
def embed_text(chunk: Chunk) -> str:
    """
    Generate an embedding string for the given chunk.

    The embedding string includes the title from the chunk's metadata, the header path, and the chunk text itself.
    """
    return f"{chunk.meta.get('title','')} — {chunk.header_path}\n\n{chunk.text}" 

#### Load the Corpus

In [4]:
DOCS_DIR = Path("Docs")
doc_paths = sorted(DOCS_DIR.glob("*.md"))
assert len(doc_paths) >= 8, f"need >=8 docs, found {len(doc_paths)} in {DOCS_DIR.resolve()}"

# Repo-relative source keeps gold labels valid if the notebook moves.
documents = [(f"week-5/Docs/{path.name}", *load_doc(path)) for path in doc_paths]

for source, meta, body in documents:
    assert meta.get("title"), f"missing frontmatter title: {source}"
    print(f"{len(body):>7,} chars  {source}")

  6,339 chars  week-5/Docs/Agent-Framework-RAG.md
 17,348 chars  week-5/Docs/Agent-hooks.md
  8,663 chars  week-5/Docs/Azure-AI-Search.md
  6,791 chars  week-5/Docs/Azure-Cosmos-DB.md
 11,718 chars  week-5/Docs/Evaluation.md
  6,067 chars  week-5/Docs/Microsoft-Agent-Framework-Overview.md
  2,980 chars  week-5/Docs/Microsoft-Foundry-evaluation.md
  8,650 chars  week-5/Docs/Neo4j.md


### 1c. Define Chunking

Three strategies, all sharing the signature `(meta, body, source) -> list[Chunk]` so they can be compared on identical queries and labels.

- **`chunk_fixed`** — Collapses whitespace and cuts the body into overlapping character windows. It is structurally blind: a window can start mid-sentence and straddle two unrelated sections. Overlap is the only defense against splitting a fact in half. Records `char_start` so each window can later be mapped back to the section it overlaps.
- **`chunk_markdown_header`(Used in part 2 with two different embedders)** — Splits on  the markdown headers of `#`/`##`/`###` so every chunk is one authored section, and captures the heading information as `header_path`. Chunks are are based off of topics as definined in the markdown headings, but are not consistent in length: a long section with several code blocks can exceed the embedding model's input window and be silently truncated.
- **`chunk_markdown_header_recursive`** — Header split first, then a recursive character split to bound each section at ~800 characters with 100 of overlap. Every piece inherits its parent section's `header_path`, so the structural metadata survives the second cut. Intended to keep the coherence of header splitting while staying inside the encoder's limit.

Headings and frontmatter `title` are prepended to the embedded text (see `embed_text`), because a chunk's body often never names the product or feature the query asks about.

In [5]:

HEADERS = [("#", "h1"), ("##", "h2"), ("###", "h3")]

def header_path_of(section) -> str:
    return " > ".join(section.metadata[level] for level in ("h1", "h2", "h3") if section.metadata.get(level))

# === Strategy 1 ===

def chunk_fixed(meta, body, source, size=320, overlap=40):
    """Baseline: header-blind character windows, whitespace collapsed."""
    if overlap >= size:
        raise ValueError("overlap must be smaller than size")
    text = re.sub(r"\s+", " ", body).strip()
    chunker_name = f"fixed-{size}-{overlap}"
    return [
        Chunk(text=text[start:start + size],
              source=source,
              chunker=chunker_name,
              ordinal=ordinal,
              header_path="",
              meta={**meta, "char_start": start})
        for ordinal, start in enumerate(range(0, len(text), size - overlap))
    ]


# === Strategy 2 ===

def chunk_markdown_header(meta, body, source):
    """One chunk per markdown section; sections are unbounded in length."""
    sections = MarkdownHeaderTextSplitter(HEADERS, strip_headers=False).split_text(body)
    return [
        Chunk(text=section.page_content,
              source=source,
              chunker="md-header",
              ordinal=ordinal,
              header_path=header_path_of(section),
              meta={**meta, **section.metadata})
        for ordinal, section in enumerate(sections)
    ]


# === Strategy 3 ===

def chunk_markdown_header_recursive(meta, body, source, size=800, overlap=100):
    """Header split first, then bound section length; sections here run long."""
    sections = MarkdownHeaderTextSplitter(HEADERS, strip_headers=False).split_text(body)
    sub_splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)

    chunks = []
    for section in sections:
        for piece in sub_splitter.split_text(section.page_content):
            chunks.append(
                Chunk(text=piece,
                      source=source,
                      chunker="md-header+rec",
                      ordinal=len(chunks),
                      header_path=header_path_of(section),
                      meta={**meta, **section.metadata})
            )
    return chunks


In [6]:
CHUNKERS = {
    "fixed-320-40": chunk_fixed,
    "md-header": chunk_markdown_header,
    "md-header+rec": chunk_markdown_header_recursive,
}

# Keys match the chunker field each function stamps on its chunks.
configs = {
    name: [chunk
           for source, meta, body in documents
           for chunk in chunker(meta, body, source)]
    for name, chunker in CHUNKERS.items()
}

for name, chunks in configs.items():
    print(f"{name:16s} -> {len(chunks):>4,} chunks")

fixed-320-40     ->  236 chunks
md-header        ->   73 chunks
md-header+rec    ->  130 chunks


### 1d. Define Embedders

Both embedders share one interface: `name`, `dim`, and an `encode` function returning L2-normalized `float32` vectors. The evaluation loop can therefore swap them without depending on provider-specific code.

- **`all-MiniLM-L6-v2` (local, 384 dimensions)**: A small sentence-transformer that runs locally on the CPU with no API cost. Its 256-token input window means long chunks may be truncated, so text near the end of an unbounded section might not contribute to the vector.
- **`text-embedding-3-small` (hosted on Microsoft Foundry, 1536 dimensions)**: A hosted model with an 8,191-token input limit. Its tradeoffs are per-token cost, network latency, and a dependency on Azure credentials, so recomputing embeddings incurs another API call unless results are cached.

Because a pgvector `vector(n)` column has a fixed dimension, the two models cannot share one vector column. This notebook stores their embeddings in separate tables, each joined to the shared `doc_chunk` table by `chunk_uid`.

In [7]:
@dataclass
class Embedder:
    name: str
    dim: int
    encode: callable   

def _normalize(vectors: np.ndarray) -> np.ndarray:
    vectors = np.asarray(vectors, dtype="float32")
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

# === 1 - Local: sentence-transformers ===
#=========================================
minilm_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

minilm = Embedder(
    name="all-MiniLM-L6-v2",
    dim=minilm_model.get_embedding_dimension(),
    encode=lambda texts: minilm_model.encode(texts, normalize_embeddings=True, batch_size=64).astype("float32"),
)



# === 2 - Hosted: Azure AI Foundry, text-embedding-3-small ===
#=============================================================
load_dotenv()

project = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)

# Embeddings route on the resource root; the project-scoped /openai/v1/ path has no /embeddings.
RESOURCE_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"].split("/api/projects")[0]


openai_client = AzureOpenAI(
    azure_endpoint=RESOURCE_ENDPOINT,
    azure_ad_token_provider=get_bearer_token_provider(
        DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
    ),
    api_version="2024-10-21",
)

# On Foundry this is the deployment name, which need not match the model name.
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")


def _openai_encode(texts, model=EMBEDDING_DEPLOYMENT, dimensions=None, batch_size=256):
    """
    Encode a list of texts into embeddings using the OpenAI API via Azure AI Foundry.

    Args:
        texts (List[str]): The texts to encode.
        model (str): The embedding model to use.
        dimensions (int, optional): The dimensionality of the embeddings.
        batch_size (int): The batch size for API requests.

    Returns:
        np.ndarray: Normalized embeddings as a NumPy array.
    """
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        kwargs = {"dimensions": dimensions} if dimensions else {}
        response = openai_client.embeddings.create(model=model, input=batch, **kwargs)
        # The API may return items out of order; index is authoritative.
        vectors.extend(item.embedding for item in sorted(response.data, key=lambda item: item.index))
    return _normalize(np.array(vectors))


openai_small = Embedder(
    name="text-embedding-3-small",
    dim=1536,
    encode=_openai_encode,
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5708.27it/s]


### 1e. Connection to local Postgres with pgvector

In [8]:
import hashlib, json, psycopg, pgvector
from pgvector.psycopg import register_vector
from psycopg import sql

# Registry: embedder name -> table. Doubles as the allow-list for identifiers below.
EMBEDDERS = [minilm, openai_small]
EMBEDDING_TABLES = {
    "all-MiniLM-L6-v2": "chunk_emb_minilm",
    "text-embedding-3-small": "chunk_emb_oai",
}


def chunk_uid(chunk):
    """
    Generate a unique identifier for a document chunk based on its source, chunker, and ordinal.

    Args:
        chunk (Chunk): The document chunk object.

    Returns:
        str: A SHA-256 hash representing the unique identifier for the chunk.
    """
    key = f"{chunk.source}|{chunk.chunker}|{chunk.ordinal}"
    return hashlib.sha256(key.encode()).hexdigest()


def embedding_table(embedder):
    """
    Resolve the pgvector table that stores a given embedder's vectors.

    Args:
        embedder (Embedder): The embedder whose table is needed.

    Returns:
        psycopg.sql.Identifier: A quoted identifier safe to compose into SQL.

    Raises:
        KeyError: If the embedder has no registered table.
    """
    return sql.Identifier(EMBEDDING_TABLES[embedder.name])


def ensure_schema(connection):
    """
    Create the chunk table and one vector table per registered embedder.

    A pgvector column fixes its dimension in the column type, so each model gets
    its own table rather than a shared table with a model discriminator column.

    Args:
        connection (psycopg.Connection): An open connection to the pgvector database.

    Returns:
        None
    """
    with connection.cursor() as cursor:
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS doc_chunk (
                chunk_uid   text PRIMARY KEY,
                source      text NOT NULL,
                chunker     text NOT NULL,
                ordinal     int  NOT NULL,
                header_path text NOT NULL DEFAULT '',
                content     text NOT NULL,
                n_chars     int  NOT NULL,
                title       text,
                url         text,
                doc_author  text,
                ms_date     text,
                meta        jsonb NOT NULL DEFAULT '{}'::jsonb
            )
        """)
        cursor.execute("CREATE INDEX IF NOT EXISTS doc_chunk_chunker_idx ON doc_chunk (chunker)")

        for embedder in EMBEDDERS:
            table = embedding_table(embedder)
            cursor.execute(sql.SQL("""
                CREATE TABLE IF NOT EXISTS {table} (
                    chunk_uid   text PRIMARY KEY REFERENCES doc_chunk(chunk_uid) ON DELETE CASCADE,
                    embed_input text NOT NULL,
                    emb         vector({dim}) NOT NULL
                )
            """).format(table=table, dim=sql.Literal(embedder.dim)))
            cursor.execute(sql.SQL(
                "CREATE INDEX IF NOT EXISTS {index} ON {table} USING hnsw (emb vector_cosine_ops)"
            ).format(
                index=sql.Identifier(f"{EMBEDDING_TABLES[embedder.name]}_emb_idx"),
                table=table,
            ))


CHUNK_INSERT = """
INSERT INTO doc_chunk (chunk_uid, source, chunker, ordinal, header_path, content,
                       n_chars, title, url, doc_author, ms_date, meta)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s::jsonb)
ON CONFLICT (chunk_uid) DO UPDATE SET
    content = EXCLUDED.content, n_chars = EXCLUDED.n_chars, meta = EXCLUDED.meta
"""


def chunk_rows(chunks):
    """
    Generate model-agnostic rows for insertion into doc_chunk.

    Args:
        chunks (list[Chunk]): The document chunks to persist.

    Yields:
        tuple: A tuple of column values matching CHUNK_INSERT.
    """
    for chunk in chunks:
        ms_date = chunk.meta.get("ms.date")
        yield (
            chunk_uid(chunk), chunk.source, chunk.chunker, chunk.ordinal,
            chunk.header_path, chunk.text, len(chunk.text),
            chunk.meta.get("title"), chunk.meta.get("Url"),
            chunk.meta.get("author"),
            str(ms_date) if ms_date is not None else None,
            json.dumps(chunk.meta, default=str),
        )


def write_chunks(connection, chunks):
    """
    Upsert chunk text and metadata, without touching any embedding table.

    Args:
        connection (psycopg.Connection): An open connection to the pgvector database.
        chunks (list[Chunk]): The document chunks to persist.

    Returns:
        int: The number of chunks written.
    """
    with connection.cursor() as cursor:
        cursor.executemany(CHUNK_INSERT, list(chunk_rows(chunks)))
    return len(chunks)


def write_embeddings(connection, embedder, chunks):
    """
    Encode chunks with one embedder and upsert the vectors into that model's table.

    Call once per embedder; adding a model requires a registry entry and nothing else.

    Args:
        connection (psycopg.Connection): An open connection to the pgvector database.
        embedder (Embedder): The embedder supplying name, dim, and encode.
        chunks (list[Chunk]): The document chunks to encode, already written to doc_chunk.

    Returns:
        int: The number of vectors written.
    """
    statement = sql.SQL("""
        INSERT INTO {table} (chunk_uid, embed_input, emb)
        VALUES (%s,%s,%s)
        ON CONFLICT (chunk_uid) DO UPDATE SET
            embed_input = EXCLUDED.embed_input, emb = EXCLUDED.emb
    """).format(table=embedding_table(embedder))

    texts = [embed_text(chunk) for chunk in chunks]
    vectors = embedder.encode(texts)
    rows = list(zip([chunk_uid(chunk) for chunk in chunks], texts, vectors, strict=True))

    with connection.cursor() as cursor:
        cursor.executemany(statement, rows)
    return len(rows)



### 1f. Add Data to local DB

In [9]:
# with psycopg.connect(os.environ["PGVECTOR_URL"]) as connection:
#     register_vector(connection)
#     ensure_schema(connection)

#     all_chunks = [chunk for chunks in configs.values() for chunk in chunks]
#     print(f"{write_chunks(connection, all_chunks):>6,} chunks")
#     for embedder in EMBEDDERS:
#         print(f"{write_embeddings(connection, embedder, all_chunks):>6,} vectors  {embedder.name}")

#     connection.commit()

### 1g. Open Local DB Connection

In [10]:
# Kept open for the rest of the notebook; close it in the last cell.
conn = psycopg.connect(os.environ["PGVECTOR_URL"], row_factory=psycopg.rows.dict_row)
register_vector(conn)

with conn.cursor() as cursor:
    for table in ["doc_chunk", *EMBEDDING_TABLES.values()]:
        count = cursor.execute(sql.SQL("SELECT count(*) AS n FROM {}").format(sql.Identifier(table))).fetchone()["n"]
        print(f"{table:20s} {count:>6,}")

doc_chunk               439
chunk_emb_minilm        439
chunk_emb_oai           439


In [11]:
def retrieve(connection, query, embedders=None, chunkers=None, k=5):
    """
    Retrieve the top-k chunks for a query from one or more embedders' vector tables.

    Args:
        connection (psycopg.Connection): An open connection to the pgvector database.
        query (str): The natural-language query.
        embedders (str | Embedder | list | None): Registered name(s) such as
            "all-MiniLM-L6-v2" or "text-embedding-3-small", Embedder objects, or None for all.
        chunkers (str | list[str] | None): Restrict to one or more chunking strategies, or None for all.
        k (int): The number of chunks to return per embedder.

    Returns:
        dict[str, list[dict]]: Embedder name -> rows of chunk_uid, source, chunker,
            header_path, content, and distance, ordered by ascending cosine distance.

    Examples:
        ```
        q = "What package do I install to use Agent Hooks?"

        retrieve(conn, q)                                                  # both models, all chunkers
        retrieve(conn, q, embedders="text-embedding-3-small")              # hosted only
        retrieve(conn, q, embedders="all-MiniLM-L6-v2", chunkers="md-header")
        retrieve(conn, q, embedders=["all-MiniLM-L6-v2", "text-embedding-3-small"],
                chunkers=["md-header", "md-header+rec"])

        rows = retrieve(conn, q, embedders="all-MiniLM-L6-v2")["all-MiniLM-L6-v2"]
        ```
    """
    by_name = {embedder.name: embedder for embedder in EMBEDDERS}

    if embedders is None:
        selected = list(EMBEDDERS)
    else:
        if isinstance(embedders, (str, Embedder)):
            embedders = [embedders]
        selected = []
        for item in embedders:
            if isinstance(item, Embedder):
                selected.append(item)
            elif item in by_name:
                selected.append(by_name[item])
            else:
                raise KeyError(f"unknown embedder {item!r}; registered: {sorted(by_name)}")

    if isinstance(chunkers, str):
        chunkers = [chunkers]
    elif chunkers is not None:
        chunkers = list(chunkers)

    results = {}
    for embedder in selected:
        statement = sql.SQL("""
            SELECT c.chunk_uid, c.source, c.chunker, c.header_path, c.content,
                   e.emb <=> %(vector)s AS distance
            FROM {table} e
            JOIN doc_chunk c USING (chunk_uid)
            WHERE %(chunkers)s::text[] IS NULL OR c.chunker = ANY(%(chunkers)s::text[])
            ORDER BY distance
            LIMIT %(k)s
        """).format(table=embedding_table(embedder))

        params = {"vector": embedder.encode([query])[0], "chunkers": chunkers, "k": k}
        with connection.cursor() as cursor:
            results[embedder.name] = cursor.execute(statement, params).fetchall()
    return results

In [12]:
print(f"==== EMBEDDERS ====")
print([(embedder.name, embedder.dim) for embedder in EMBEDDERS])
print()
print(f"==== CHUNKERS ====")
print(list(CHUNKERS.keys()))
print()

==== EMBEDDERS ====
[('all-MiniLM-L6-v2', 384), ('text-embedding-3-small', 1536)]

==== CHUNKERS ====
['fixed-320-40', 'md-header', 'md-header+rec']



In [13]:
q = "What package do I install to use Agent Hooks?"
retrieve(conn, q, embedders="all-MiniLM-L6-v2", chunkers="md-header", k=3)

{'all-MiniLM-L6-v2': [{'chunk_uid': '0f2ae5cd45be1e37660fee2a655cf4611cb88ccfba9eb76d57bc25c4547acd43',
   'source': 'week-5/Docs/Agent-hooks.md',
   'chunker': 'md-header',
   'header_path': 'Agent hooks | Microsoft Learn > Install Agent Hooks',
   'content': "## Install Agent Hooks  \nInstall the Agent Hooks SDK as a direct dependency:  \n```bash\npip install agent-hooks-sdk\n```  \nIf you use `uv`:  \n```bash\nuv add agent-hooks-sdk\n```  \nThe `agent-hooks-sdk` dependency is lazy-imported. Importing `agent_framework` doesn't load the SDK unless you create an Agent Hooks middleware bundle.  \nNote  \n`agent-framework-core` doesn't include an `agent-hooks` extra. Install `agent-hooks-sdk` separately before you create an Agent Hooks middleware bundle.",
   'distance': 0.3060597388130476},
  {'chunk_uid': 'b8b2f185918826741da0658b2b1f48bb0da933bff1a4f21e2262c86ed88331a2',
   'source': 'week-5/Docs/Agent-hooks.md',
   'chunker': 'md-header',
   'header_path': 'Agent hooks | Microsoft Le

### 1h. Visualize Database Table Schemas (Illustrative not Exact)

#### `doc_chunk` (one row per chunk, shared across chunkers/embedders)

| chunk_uid | source | chunker | ordinal | header_path | content | n_chars | title | url | doc_author | ms_date | meta |
|---|---|---|---|---|---|---|---|---|---|---|---|
| `9f3a1` | week-5/Docs/Agent-hooks.md | md-header | 2 | `> Install Agent Hooks` | `## Install...` | 64 | Agent hooks | `.../agent-hooks` | — | 2025-08-01 | `{...}` |
| `4b7e2` | week-5/Docs/Agent-hooks.md | md-header | 4 | `> Configure enforcement` | `## Configure...` | 218 | Agent hooks | `.../agent-hooks` | — | 2025-08-01 | `{...}` |
| `c02d8` | week-5/Docs/Evaluation.md | md-header | 6 | `> Conversation split strategies` | `## Conversation...` | 341 | Evaluation | `.../evaluation` | — | 2025-07-15 | `{...}` |
| `7a19f` | week-5/Docs/Microsoft-Foundry-evaluation.md | md-header | 3 | `> Quality gates` | `## Quality...` | 179 | Microsoft Foundry evaluation | `.../foundry-eval` | — | 2025-09-02 | `{...}` |
| `e6c34` | week-5/Docs/Neo4j.md | md-header | 5 | `> ...knowledge graph > Installation` | `### Installation...` | 58 | Neo4j | `.../neo4j` | — | 2025-06-20 | `{...}` |

#### `chunk_emb_minilm` (`vector(384)`, one row per `chunk_uid` for this embedder)

| chunk_uid | embed_input | emb |
|---|---|---|
| `9f3a1` | `Agent hooks — Install Agent Hooks\n\npip install...` | `[0.03...` |
| `4b7e2` | `Agent hooks — Configure enforcement\n\n...` | `[-0.1...` |
| `c02d8` | `Evaluation — Conversation split strategies\n\n...` | `[0.07...` |
| `7a19f` | `Microsoft Foundry evaluation — Quality gates\n\nPin...` | `[0.02...` |
| `e6c34` | `Neo4j — ...Installation\n\npip install...` | `[-0.0...` |

#### `chunk_emb_oai` (`vector(1536)`, `text-embedding-3-small`)

| chunk_uid | embed_input | emb |
|---|---|---|
| `9f3a1` | `Agent hooks — Install Agent Hooks\n\npip install...` | `[0.01...` |
| `4b7e2` | `Agent hooks — Configure enforcement\n\n...` | `[0.04...` |
| `c02d8` | `Evaluation — Conversation split strategies\n\n...` | `[-0.0...` |
| `7a19f` | `Microsoft Foundry evaluation — Quality gates\n\nPin...` | `[0.05...` |
| `e6c34` | `Neo4j — ...Installation\n\npip install...` | `[0.00...` |

## Part 2: Evaluate retrieval across embedding models

Part 1 created three chunking strategies and two embedding configurations. To isolate the embedding model as the comparison variable, this section evaluates both `all-MiniLM-L6-v2` (384 dimensions) and `text-embedding-3-small` (1536 dimensions) using only the `md-header` chunking strategy. Because this strategy splits at authored Markdown headings without imposing a second size limit, chunk lengths vary.

The database uses three tables for this comparison:
- `doc_chunk`: stores shared chunk text and metadata.
- `chunk_emb_minilm`: stores the 384-dimensional MiniLM embeddings.
- `chunk_emb_oai`: stores the 1536-dimensional `text-embedding-3-small` embeddings.

### 2a. Test queries and facts
* I created the QA first to show which document should contain the answer. 

In [14]:
QA = [
    {"query": "What function lets an agent search a vector store?",
     "fact": "create_vector_search_tool",
     "source": "week-5/Docs/Agent-Framework-RAG.md"},

    {"query": "How do I turn on hybrid keyword search for a search tool?",
     "fact": "keyword_hybrid",
     "source": "week-5/Docs/Agent-Framework-RAG.md"},

    {"query": "What package do I install to use Agent Hooks?",
     "fact": "agent-hooks-sdk",
     "source": "week-5/Docs/Agent-hooks.md"},

    {"query": "How long does an Agent Hooks interceptor wait before it times out?",
     "fact": "five seconds",
     "source": "week-5/Docs/Agent-hooks.md"},

    {"query": "Which modes of retrieval are supported by AzureAISearchContextProvider?",
     "fact": "semantic and agentic retrieval",
     "source": "week-5/Docs/Azure-AI-Search.md"},

    {"query": "Which Cosmos DB provider contains memories of facts or summaries between sessions?",
     "fact": "CosmosMemoryContextProvider",
     "source": "week-5/Docs/Azure-Cosmos-DB.md"},

    {"query": "Which evaluator runs checks locally without calling an API?",
     "fact": "LocalEvaluator",
     "source": "week-5/Docs/Evaluation.md"},

    {"query": "Which agent comes with planning and todo tracking built in?",
     "fact": "Harness Agent",
     "source": "week-5/Docs/Microsoft-Agent-Framework-Overview.md"},

    {"query": "What should I pin so evaluation results stay comparable across runs?",
     "fact": "datasets, model deployments, evaluator versions, and rubric versions",
     "source": "week-5/Docs/Microsoft-Foundry-evaluation.md"},

    {"query": "What package do I install for the Neo4j GraphRAG context provider?",
    "fact": "agent-framework-neo4j",
    "source": "week-5/Docs/Neo4j.md"},
]

qa = [(item["query"], item["fact"]) for item in QA]

#### Where to validate each answer

Each fact string appears in only one document, so a substring hit is unambiguous.

| # | Document | Section / header path | What you will see |
|---|---|---|---|
| 1 | `Agent-Framework-RAG.md` | intro paragraph under the H1 | "provides native vector-store contracts and `create_vector_search_tool()`" |
| 2 | `Agent-Framework-RAG.md` | `Customize search behavior` → options table, `search_type` row | "Selects `vector` or `keyword_hybrid` search" |
| 3 | `Agent-hooks.md` | `Install Agent Hooks` | "pip install agent-hooks-sdk" |
| 4 | `Agent-hooks.md` | `Configure enforcement` → parameters table, `timeout` row | "The default is five seconds." |
| 5 | `Azure-AI-Search.md` | `Azure AI Search` | "In Python, `AzureAISearchContextProvider` supports semantic and agentic retrieval. |
| 6 | `Azure-Cosmos-DB.md` | pattern table under the H1, and `Configure the memory provider` | "`CosmosMemoryContextProvider` (Python) \| Extracts facts, procedural knowledge, episodic memories…" |
| 7 | `Evaluation.md` | `Local evaluators` | "`LocalEvaluator` runs checks locally without API calls" |
| 8 | `Microsoft-Agent-Framework-Overview.md` | four-areas table under the H1 | "**Harness Agent** \| An opinionated agent with batteries-included capabilities…" |
| 9 | `Microsoft-Foundry-evaluation.md` | `Quality gates` | "Pin datasets, model deployments, evaluator versions, and rubric versions when results must be comparable across runs." |
| 10 | `Neo4j.md` | `GraphRAG from an existing knowledge graph` → `Installation` | "pip install agent-framework-neo4j" |


### 2b. Find candidate chunks for scoring
Search for each labeled fact in chunks created by the `md-header` strategy.

In [15]:
import textwrap

CANDIDATE_SQL = """
SELECT 
    chunk_uid,
    source,
    header_path,
    n_chars,
       -- Window centered on the match, not the top of the chunk.
    substring(content from greatest(strpos(lower(content), lower(%(fact)s)) - 70, 1) for 180) AS snippet
FROM doc_chunk
WHERE chunker = 'md-header'
  AND strpos(lower(content), lower(%(fact)s)) > 0
ORDER BY source, ordinal
"""

def show_candidates(connection, item):
    """
    Show candidate document chunks for a given query item.
    This searches for document chunks that contain the specified fact using a SQL query.
    Args:
        connection: A database connection object.
        item: A dictionary containing the query and fact to search for.
    """
    
    with connection.cursor() as cursor:
        rows = cursor.execute(CANDIDATE_SQL, {"fact": item["fact"]}).fetchall()

    print("=" * 100)
    print(f"Q: {item['query']}")
    print(f"   fact: {item['fact']!r}   candidates: {len(rows)}")
    for n, row in enumerate(rows, 1):
        snippet = re.sub(r"\s+", " ", row["snippet"]).strip()
        print(f"\n  [{n}] {row['source'].split('/')[-1]}  ({row['n_chars']:,} chars)")
        print(f"      {row['header_path'] or '(no header)'}")
        print(textwrap.fill(f"...{snippet}...", width=92,
                            initial_indent="      ", subsequent_indent="      "))
    print()

for item in QA:
    show_candidates(conn, item)

Q: What function lets an agent search a vector store?
   fact: 'create_vector_search_tool'   candidates: 3

  [1] Agent-Framework-RAG.md  (940 chars)
      RAG | Microsoft Learn
      ...graph). Agent Framework provides native vector-store contracts and
      `create_vector_search_tool()`. The helper turns any `SupportsVectorSearch`
      implementation into a function tool,...

  [2] Agent-Framework-RAG.md  (2,898 chars)
      RAG | Microsoft Learn > Create a native vector search tool
      ...nt, Filter, FilterGroup, InMemoryCollection, Param, VectorStoreField,
      create_vector_search_tool, vectorstoremodel, ) from agent_framework.openai import
      OpenAIChatClient, OpenAIEmbe...

  [3] Agent-Framework-RAG.md  (1,301 chars)
      RAG | Microsoft Learn > Customize search behavior
      ...### Customize search behavior Configure `create_vector_search_tool()` with the
      following options: | Option | Purpose | | --- | --- | | `name` | Sets the function
      name exposed t...

Q

### 2c. Relevance labeling: procedure and rationale

**Procedure**
1. `show_candidates` ran the fact-substring search (`CANDIDATE_SQL`) for all ten queries against the `md-header` chunks. It printed every matching chunk in source order, with a snippet centered on the fact.
2. I read each candidate and labeled it **Relevant** when it directly answered the query or **Partially relevant** when it contained the fact only as supporting context, such as an import, code sample, or resource link.
3. For scoring, I combined both labels into an **expanded relevance set**. A chunk belongs to this set when it is an `md-header` chunk from the expected source document and contains the labeled fact string. No additional manual exclusion is applied.

**Rationale**
- Most queries produced exactly one candidate under `md-header` chunking. A few facts also appeared in code examples, imports, or resource links elsewhere in the same document.
- Including partially relevant chunks creates multi-chunk relevance sets for four queries, making recall more informative than it would be if every query had only one relevant chunk.
- The rule is reproducible from one SQL query instead of depending on a hand-maintained include/exclude list.
- This is a deliberately permissive definition. Reported precision and recall measure retrieval of fact-bearing chunks in the expected document, not only chunks that independently provide the best answer. The separate `PRIMARY_RELEVANT` labels retain that stricter distinction.

**Relevance labels by query (both labels belong to the expanded relevance set)**

1. **What function lets an agent search a vector store?** - fact: `create_vector_search_tool`
   - Relevant
     - RAG | Microsoft Learn
   - Partially relevant
     - RAG | Microsoft Learn > Create a native vector search tool
     - RAG | Microsoft Learn > Customize search behavior
2. **How do I turn on hybrid keyword search for a search tool?** - fact: `keyword_hybrid`
   - Relevant
     - RAG | Microsoft Learn > Customize search behavior
3. **What package do I install to use Agent Hooks?** - fact: `agent-hooks-sdk`
   - Relevant
     - Agent hooks | Microsoft Learn > Install Agent Hooks
4. **How long does an Agent Hooks interceptor wait before it times out?** - fact: `five seconds`
   - Relevant
     - Agent hooks | Microsoft Learn > Configure enforcement
5. **Which modes of retrieval are supported by AzureAISearchContextProvider?** - fact: `semantic and agentic retrieval`
   - Relevant
     - Azure AI Search | Microsoft Learn
6. **Which Cosmos DB provider contains memories of facts or summaries between sessions?** - fact: `CosmosMemoryContextProvider`
   - Relevant
     - Azure Cosmos DB | Microsoft Learn
   - Partially relevant
     - Azure Cosmos DB | Microsoft Learn > Add long-term semantic memory > Configure the memory provider
7. **Which evaluator runs checks locally without calling an API?** - fact: `LocalEvaluator`
   - Relevant
     - Evaluation | Microsoft Learn > Local evaluators
   - Partially relevant
     - Evaluation | Microsoft Learn > Core concepts
     - Evaluation | Microsoft Learn > Local evaluators > Built-in checks
     - Evaluation | Microsoft Learn > Local evaluators > Custom function evaluators
     - Evaluation | Microsoft Learn > Mix multiple evaluators
8. **Which agent comes with planning and todo tracking built in?** - fact: `Harness Agent`
   - Relevant
     - Microsoft Agent Framework Overview | Microsoft Learn
9. **What should I pin so evaluation results stay comparable across runs?** - fact: `datasets, model deployments, evaluator versions, and rubric versions`
   - Relevant
     - Microsoft Foundry evaluation | Microsoft Learn > Quality gates
10. **What package do I install for the Neo4j GraphRAG context provider?** - fact: `agent-framework-neo4j`
    - Relevant
      - Neo4j | Microsoft Learn > GraphRAG from an existing knowledge graph > Installation
    - Partially relevant
      - Neo4j | Microsoft Learn > GraphRAG from an existing knowledge graph > Resources (link to the package page, not the install command)

In [16]:
# Relevant = every md-header chunk in the source doc that contains the fact string.
RELEVANT_SQL = """
SELECT chunk_uid, header_path
FROM doc_chunk
WHERE chunker = 'md-header'
  AND source  = %(source)s
  AND strpos(lower(content), lower(%(fact)s)) > 0
"""

def relevant_rows(connection, item):
    """Same query as relevant_chunks, kept as rows so header_path stays visible."""
    with connection.cursor() as cursor:
        return cursor.execute(RELEVANT_SQL, {"source": item["source"], "fact": item["fact"]}).fetchall()

print(f"==== Relevant Chunks per Query ====")
print()
for item in QA:
    rows = relevant_rows(conn, item)
    print(f"{len(rows):>2}  {item['query'][:60]}")
    for row in rows:
        print(f"      - {row['header_path']}")

==== Relevant Chunks per Query ====

 3  What function lets an agent search a vector store?
      - RAG | Microsoft Learn
      - RAG | Microsoft Learn > Create a native vector search tool
      - RAG | Microsoft Learn > Customize search behavior
 1  How do I turn on hybrid keyword search for a search tool?
      - RAG | Microsoft Learn > Customize search behavior
 1  What package do I install to use Agent Hooks?
      - Agent hooks | Microsoft Learn > Install Agent Hooks
 1  How long does an Agent Hooks interceptor wait before it time
      - Agent hooks | Microsoft Learn > Configure enforcement
 1  Which modes of retrieval are supported by AzureAISearchConte
      - Azure AI Search | Microsoft Learn
 2  Which Cosmos DB provider contains memories of facts or summa
      - Azure Cosmos DB | Microsoft Learn > Add long-term semantic memory > Configure the memory provider
      - Azure Cosmos DB | Microsoft Learn
 5  Which evaluator runs checks locally without calling an API?
      - Eval

In [17]:
RELEVANT = {
    item["query"]: [(row["chunk_uid"], row["header_path"]) for row in relevant_rows(conn, item)]
    for item in QA
}
relevant_df = pd.DataFrame(
    [(query[:55], chunk_uid[:5] + "...", header_path)
     for query, rows in RELEVANT.items()
     for chunk_uid, header_path in rows],
    columns=["query", "chunk_uid", "header_path"],
)
relevant_df["query"] = relevant_df["query"].mask(relevant_df["query"].duplicated(), "")  # blank repeats, like a grouped table
print("==== Relevant score card ====")
print()
relevant_df

==== Relevant score card ====



,query,chunk_uid,header_path
0,What function lets an agent search a vector st...,e0c35...,RAG | Microsoft Learn
1,,b9d1b...,RAG | Microsoft Learn > Create a native vector...
2,,168f5...,RAG | Microsoft Learn > Customize search behavior
3,How do I turn on hybrid keyword search for a s...,168f5...,RAG | Microsoft Learn > Customize search behavior
4,What package do I install to use Agent Hooks?,0f2ae...,Agent hooks | Microsoft Learn > Install Agent ...
5,How long does an Agent Hooks interceptor wait ...,33018...,Agent hooks | Microsoft Learn > Configure enfo...
6,Which modes of retrieval are supported by Azur...,9615a...,Azure AI Search | Microsoft Learn
7,Which Cosmos DB provider contains memories of ...,0813c...,Azure Cosmos DB | Microsoft Learn > Add long-t...
8,,564c7...,Azure Cosmos DB | Microsoft Learn
9,Which evaluator runs checks locally without ca...,8563d...,Evaluation | Microsoft Learn > Core concepts


### 2d. Compare embedding models against the score card

In [18]:
# Below I make a list of the primary relevant header paths in my documents for each query.
PRIMARY_RELEVANT = {
    "What function lets an agent search a vector store?":
        "RAG | Microsoft Learn",
    "How do I turn on hybrid keyword search for a search tool?":
        "RAG | Microsoft Learn > Customize search behavior",
    "What package do I install to use Agent Hooks?":
        "Agent hooks | Microsoft Learn > Install Agent Hooks",
    "How long does an Agent Hooks interceptor wait before it times out?":
        "Agent hooks | Microsoft Learn > Configure enforcement",
    "Which modes of retrieval are supported by AzureAISearchContextProvider?":
        "Azure AI Search | Microsoft Learn",
    "Which Cosmos DB provider contains memories of facts or summaries between sessions?":
        "Azure Cosmos DB | Microsoft Learn",
    "Which evaluator runs checks locally without calling an API?":
        "Evaluation | Microsoft Learn > Local evaluators",
    "Which agent comes with planning and todo tracking built in?":
        "Microsoft Agent Framework Overview | Microsoft Learn",
    "What should I pin so evaluation results stay comparable across runs?":
        "Microsoft Foundry evaluation | Microsoft Learn > Quality gates",
    "What package do I install for the Neo4j GraphRAG context provider?":
        "Neo4j | Microsoft Learn > GraphRAG from an existing knowledge graph > Installation",
}

def primary_chunk_uid(query):
    """Resolve chunk_uid for a query's single 'Relevant' header_path, via the existing RELEVANT rows."""
    target = PRIMARY_RELEVANT[query]
    return next(uid for uid, header_path in RELEVANT[query] if header_path == target)


def score_query_primary_chunk(connection, item, k=5, chunker="md-header"):
    """
    Compute chunk-level precision and recall for one query, per embedder.
    ...
    """
    relevant_uids = {chunk_uid for chunk_uid, _ in RELEVANT[item["query"]]}
    primary_uid = primary_chunk_uid(item["query"])
    retrieved = retrieve(connection, item["query"], chunkers=chunker, k=k)

    rows = []
    for embedder_name, hits in retrieved.items():
        retrieved_uids = {row["chunk_uid"] for row in hits}
        true_positives = retrieved_uids & relevant_uids
        rows.append({
            "query": item["query"],
            "embedder": embedder_name,
            "k": k,
            "n_relevant": len(relevant_uids),
            "n_retrieved": len(retrieved_uids),
            "n_hits": len(true_positives),
            "precision": len(true_positives) / len(retrieved_uids) if retrieved_uids else 0.0,
            "recall": len(true_positives) / len(relevant_uids) if relevant_uids else 0.0,
            "hit_primary": primary_uid in retrieved_uids,
        })
    return rows

def score_all_queries_primary_chunk(connection, qa_items=QA, k=5, chunker="md-header"):
    """Long-format precision/recall rows for every query and embedder, ready for a DataFrame."""
    return [row for item in qa_items for row in score_query_primary_chunk(connection, item, k=k, chunker=chunker)]



#### Performance at K=5 

In [19]:
k5_scores = pd.DataFrame(score_all_queries_primary_chunk(conn, k=5))
k5_scores

,query,embedder,k,n_relevant,n_retrieved,n_hits,precision,recall,hit_primary
0,What function lets an agent search a vector st...,all-MiniLM-L6-v2,5,3,5,2,0.4,0.666667,True
1,What function lets an agent search a vector st...,text-embedding-3-small,5,3,5,3,0.6,1.000000,True
2,How do I turn on hybrid keyword search for a s...,all-MiniLM-L6-v2,5,1,5,1,0.2,1.000000,True
3,How do I turn on hybrid keyword search for a s...,text-embedding-3-small,5,1,5,1,0.2,1.000000,True
4,What package do I install to use Agent Hooks?,all-MiniLM-L6-v2,5,1,5,1,0.2,1.000000,True
5,What package do I install to use Agent Hooks?,text-embedding-3-small,5,1,5,1,0.2,1.000000,True
6,How long does an Agent Hooks interceptor wait ...,all-MiniLM-L6-v2,5,1,5,1,0.2,1.000000,True
7,How long does an Agent Hooks interceptor wait ...,text-embedding-3-small,5,1,5,1,0.2,1.000000,True
8,Which modes of retrieval are supported by Azur...,all-MiniLM-L6-v2,5,1,5,1,0.2,1.000000,True
9,Which modes of retrieval are supported by Azur...,text-embedding-3-small,5,1,5,1,0.2,1.000000,True


For the six queries with one relevant chunk, both models produced the same k=5 scores: each returned five chunks and included the one relevant chunk, yielding precision 0.20 and recall 1.00. The next table focuses on the four queries with multiple chunks in the expanded relevance set.

In [20]:
multi_relevant_df_k5 = k5_scores[k5_scores["n_relevant"] > 1]
multi_relevant_df_k5

,query,embedder,k,n_relevant,n_retrieved,n_hits,precision,recall,hit_primary
0,What function lets an agent search a vector st...,all-MiniLM-L6-v2,5,3,5,2,0.4,0.666667,True
1,What function lets an agent search a vector st...,text-embedding-3-small,5,3,5,3,0.6,1.000000,True
10,Which Cosmos DB provider contains memories of ...,all-MiniLM-L6-v2,5,2,5,1,0.2,0.500000,True
11,Which Cosmos DB provider contains memories of ...,text-embedding-3-small,5,2,5,2,0.4,1.000000,True
12,Which evaluator runs checks locally without ca...,all-MiniLM-L6-v2,5,5,5,2,0.4,0.400000,True
13,Which evaluator runs checks locally without ca...,text-embedding-3-small,5,5,5,4,0.8,0.800000,True
18,What package do I install for the Neo4j GraphR...,all-MiniLM-L6-v2,5,2,5,2,0.4,1.000000,True
19,What package do I install for the Neo4j GraphR...,text-embedding-3-small,5,2,5,2,0.4,1.000000,True


At k=5, `text-embedding-3-small` matched or outperformed MiniLM on all four multi-relevant queries. It recovered more of the expanded relevance set for the vector-search, Cosmos DB, and local-evaluator queries, while the models tied on the Neo4j query.

#### Performance at K=1

In [21]:
k1_scores = pd.DataFrame(score_all_queries_primary_chunk(conn, k=1))
k1_scores


,query,embedder,k,n_relevant,n_retrieved,n_hits,precision,recall,hit_primary
0,What function lets an agent search a vector st...,all-MiniLM-L6-v2,1,3,1,1,1.0,0.333333,False
1,What function lets an agent search a vector st...,text-embedding-3-small,1,3,1,1,1.0,0.333333,False
2,How do I turn on hybrid keyword search for a s...,all-MiniLM-L6-v2,1,1,1,1,1.0,1.000000,True
3,How do I turn on hybrid keyword search for a s...,text-embedding-3-small,1,1,1,1,1.0,1.000000,True
4,What package do I install to use Agent Hooks?,all-MiniLM-L6-v2,1,1,1,1,1.0,1.000000,True
5,What package do I install to use Agent Hooks?,text-embedding-3-small,1,1,1,1,1.0,1.000000,True
6,How long does an Agent Hooks interceptor wait ...,all-MiniLM-L6-v2,1,1,1,1,1.0,1.000000,True
7,How long does an Agent Hooks interceptor wait ...,text-embedding-3-small,1,1,1,0,0.0,0.000000,False
8,Which modes of retrieval are supported by Azur...,all-MiniLM-L6-v2,1,1,1,1,1.0,1.000000,True
9,Which modes of retrieval are supported by Azur...,text-embedding-3-small,1,1,1,1,1.0,1.000000,True


#### Analysis of Embedding Models

I scored retrieval against an expanded relevance set: an `md-header` chunk counts as relevant when it comes from the expected source document and contains the labeled fact string. This reproducible rule includes both chunks that directly answer the question and chunks that contain the fact only as supporting context. Six of the ten queries have one relevant chunk; the other four have multiple relevant chunks because the fact also appears in code examples, imports, or resource links.

At k=5, `text-embedding-3-small` matched or outperformed `all-MiniLM-L6-v2` on the four queries with multiple relevant chunks. The models tied on all six queries with one relevant chunk. At k=1, however, the results were mixed: the models did not tie on every single-relevant query, and MiniLM returned the designated primary chunk slightly more often.

##### Analysis of Performance at K=5

Retrieval returns five chunks for every query, so the maximum possible precision under this relevance definition is `min(n_relevant, k) / k`. For the six queries where only one chunk is relevant, that maximum is `1/5 = 0.20`; both models achieved precision 0.20, recall 1.00, and `hit_primary = True` on all six. These metrics therefore do not distinguish the models on those queries at k=5.

The models separate on three of the four queries with multiple relevant chunks and tie on the fourth:

| Query | Relevant chunks | `all-MiniLM-L6-v2` recall | `text-embedding-3-small` recall |
|---|---|---|---|
| vector-search-tool function | 3 | 0.67 | 1.00 |
| Cosmos DB memory provider | 2 | 0.50 | 1.00 |
| local evaluator | 5 | 0.40 | 0.80 |
| Neo4j package | 2 | 1.00 | 1.00 |

Averaged over these four queries, `text-embedding-3-small` recovered about 95% of each relevant set, compared with about 64% for `all-MiniLM-L6-v2`. Precision shows the same pattern: OpenAI's embeddings reached the maximum possible precision on the vector-search, Cosmos DB, and Neo4j queries and came close on the evaluator query. Across all ten queries, mean precision was 0.26 for MiniLM and 0.34 for OpenAI; mean recall was 0.86 and 0.98, respectively. The `hit_primary` column measures whether retrieval included the single chunk I designated as the most direct answer. Both models achieved `hit_primary = True` for all ten queries at k=5, so they differed in how much of the expanded relevant set they recovered, not in whether the primary chunk appeared somewhere in the top five.

##### Analysis of Performance at K=1

With one retrieved chunk, precision is binary: it is 1.0 when that chunk belongs to the expanded relevant set and 0.0 otherwise. Recall is `1 / n_relevant` for a relevant hit and 0.0 for a miss, so it can be fractional when a query has multiple relevant chunks. `hit_primary` is a separate binary measure that is true only when the retrieved chunk is the designated primary chunk. These three values coincide only for queries that have exactly one relevant chunk.

Using expanded-set relevance, MiniLM returned a relevant chunk for 9 of 10 queries, while `text-embedding-3-small` did so for 8 of 10. Using the stricter `hit_primary` measure, MiniLM returned the intended primary chunk for 7 of 10 queries versus 6 of 10 for `text-embedding-3-small`. The hosted model missed the only relevant chunk for the Agent Hooks timeout and Harness Agent queries, while MiniLM missed the only relevant chunk for the evaluation reproducibility query. For the Cosmos DB and local-evaluator queries, both models returned the primary chunk, producing recall of 0.50 and 0.20 because those expanded relevance sets contain two and five chunks. For the vector-search and Neo4j queries, both models returned a chunk in the expanded relevant set but not the designated primary chunk. With only ten queries, the difference is small and should be treated as descriptive evidence for this test set rather than a general ranking of the models.

## Part 3: Use LLM + RAG to Generate Grounded Answer 
Build a grounded prompt from the retrieved chunks and answer.

In [22]:
from prompts.loader import load

RAG_PROMPT = load("rag-answer")

def format_context(hits, char_limit=1200):
    """
    Number retrieved chunks so the model can cite them, and cap unbounded md-header chunks.
    Args:
        hits (list): Retrieved chunks from the database.
        char_limit (int): Maximum number of characters to include from each chunk.

    Returns:
        str: Formatted context string with numbered chunks.
    
    """
    lines = []
    for n, hit in enumerate(hits, 1):
        source = hit["source"].split("/")[-1]
        header = hit["header_path"] or "(no header)"
        lines.append(f"[{n}] {source} — {header}\n{hit['content'][:char_limit]}")
    return "\n\n".join(lines)


def generate_answer(connection, query, embedder="text-embedding-3-small", chunker="md-header", k=5):
    """
    Retrieve grounding chunks for a query, then ask the Foundry-hosted chat model
    to answer using only that context.

    Args:
        connection (psycopg.Connection): An open connection to the pgvector database.
        query (str): The natural-language question.
        embedder (str): Registered embedder name to retrieve with.
        chunker (str): Chunking strategy to restrict retrieval to.
        k (int): Number of chunks to retrieve.

    Returns:
        dict: "answer" text and "citations" (source, header_path) pairs for the retrieved chunks.
    """
    # Use the retrieve function to get the top-k relevant chunks for the query.
    hits = retrieve(connection, query, embedders=embedder, chunkers=chunker, k=k)[embedder]
    messages = [{"role": "user", "content": f"Context:\n{format_context(hits)}\n\nQuestion: {query}"}]

    try:
        openai = project.get_openai_client()
        response = openai.responses.create(
            model=RAG_PROMPT.model,
            instructions=RAG_PROMPT.text,
            input=messages,
            **RAG_PROMPT.params,
        )
        answer_text = response.output_text
    except Exception as error:
        answer_text = f"[GENERATION-SKIPPED] {error}"

    return {"answer": answer_text, "citations": [(h["source"], h["header_path"]) for h in hits]}


def show_answer(connection, query, embedder="text-embedding-3-small", chunker="md-header", k=5):
    """
    Print a query, its grounded answer, and which retrieved chunks the answer actually cites.

    Args:
        connection (psycopg.Connection): An open connection to the pgvector database.
        query (str): The natural-language question.
        embedder (str): Registered embedder name to retrieve with.
        chunker (str): Chunking strategy to restrict retrieval to.
        k (int): Number of chunks to retrieve.
    """
    result = generate_answer(connection, query, embedder=embedder, chunker=chunker, k=k)
    answer = result["answer"]
    citations = result["citations"]
    # [n] in the answer text is the model's own citation, not every retrieved chunk.
    cited_numbers = sorted(n for n in {int(m) for m in re.findall(r"\[(\d+)\]", answer)} if 1 <= n <= len(citations))

    def print_refs(numbers):
        if not numbers:
            print("    (none)")
        for n in numbers:
            source, header_path = citations[n - 1]
            print(f"    [{n}] {source.split('/')[-1]} — {header_path or '(no header)'}")

    print("=" * 100)
    print(f"Q: {query}")
    print()
    print(textwrap.fill(f"A: {answer}", width=92, initial_indent="  ", subsequent_indent="  "))
    print()
    print("  Retrieved:")
    print_refs(range(1, len(citations) + 1))
    print("  Used:")
    print_refs(cited_numbers)
    print()


In [23]:
show_answer(conn, "What package do I install to use Agent Hooks?", k=2)


Q: What package do I install to use Agent Hooks?

  A: To use Agent Hooks, you need to install the `agent-hooks-sdk` package [1].

  Retrieved:
    [1] Agent-hooks.md — Agent hooks | Microsoft Learn > Install Agent Hooks
    [2] Agent-hooks.md — Agent hooks | Microsoft Learn
  Used:
    [1] Agent-hooks.md — Agent hooks | Microsoft Learn > Install Agent Hooks



## Part 4 - Find one failure and explain it

### The failure

* Query: **"What should I pin so evaluation results stay comparable across runs?"** 
* Embedder: `all-MiniLM-L6-v2`
* Chunker: `md-header` 
* k=1.


In [24]:
failure_query = "What should I pin so evaluation results stay comparable across runs?"
print("==== all-MiniLM-L6-v2 ====")
print()
show_answer(connection=conn,query=failure_query,embedder="all-MiniLM-L6-v2",chunker="md-header", k=1)
print()
print()
print("==== text-embedding-3-small ====")
print()
show_answer(connection=conn,query=failure_query,embedder="text-embedding-3-small",chunker="md-header", k=1)
print()
print()




==== all-MiniLM-L6-v2 ====

Q: What should I pin so evaluation results stay comparable across runs?

  A: I don't know based on the retrieved context.

  Retrieved:
    [1] Evaluation.md — Evaluation | Microsoft Learn > Conversation split strategies
  Used:
    (none)



==== text-embedding-3-small ====

Q: What should I pin so evaluation results stay comparable across runs?

  A: You should pin datasets, model deployments, evaluator versions, and rubric versions to
  ensure evaluation results remain comparable across runs [1].

  Retrieved:
    [1] Microsoft-Foundry-evaluation.md — Microsoft Foundry evaluation | Microsoft Learn > Quality gates
  Used:
    [1] Microsoft-Foundry-evaluation.md — Microsoft Foundry evaluation | Microsoft Learn > Quality gates





### Likely cause

`all-MiniLM-L6-v2` retrieves `Evaluation.md > Conversation split strategies`, which discusses splitting multi-turn conversations into query/response halves. The primary chunk is `Microsoft-Foundry-evaluation.md > Quality gates`, in a different document. Both chunks concern evaluation and share broad vocabulary such as `evaluator`, `results`, and `run`, while the query's discriminating term, `pin`, appears in the primary chunk.

The result demonstrates a ranking difference between the two embedding models: at k=1, MiniLM ranks the topically similar conversation-splitting chunk first, whereas `text-embedding-3-small` ranks the `Quality gates` chunk first. This experiment does not isolate why the models differ; vector dimensionality, training data, and model objectives may all contribute. The `md-header` chunk itself is well formed and contains the complete fact, so the observed failure is a dense-retrieval ranking error rather than evidence that chunking split or omitted the answer.

### Proposed mitigations

1. **Evaluate hybrid keyword and dense retrieval.** The query's discriminating term, `pin`, appears verbatim in the primary chunk but not in the incorrectly ranked `Conversation split strategies` chunk. A BM25 component would reward that lexical overlap and could improve the ranking when combined with dense similarity. This is a proposed mitigation; I did not implement or test BM25 in this notebook.
2. **Retrieve more than one chunk.** At k=5, MiniLM achieves `hit_primary = True` for this query, showing that the primary chunk is present but ranked below another evaluation-related chunk. Retrieving the top three to five chunks gives the generation model access to the answer, although larger context can also introduce distractions and does not guarantee that the model will select the correct evidence.

In [25]:
conn.close() # Close the database connection